# Top Feature-Importance LGBM Feature Sets

Rank the sparse-filtered row-wise feature space from notebook `06_add_features_lgbm.ipynb` by LightGBM feature importance, then evaluate datasets that keep only the top `100`, `250`, and `500` features.

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.loader import Loader
from src.modeling import build_lgbm_regressor
from src.preprocessing import FeaturePreprocessor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
SPARSITY_THRESHOLD = 0.9875
TOP_K_VALUES = [100, 250, 500]

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

Feature selection must be fitted on the training split only. The holdout test split is used only for the final comparison of each selected feature set.

In [6]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

preprocessor = FeaturePreprocessor(
    zero_share_threshold=SPARSITY_THRESHOLD,
    add_rowwise=True,
)
X_aug_train = preprocessor.fit_transform(X_train_raw)
X_aug_test = preprocessor.transform(X_test_raw)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

pd.DataFrame(
    {
        "metric": ["sparsity_threshold", "base_features_kept", "added_rowwise_features", "final_features"],
        "value": [
            SPARSITY_THRESHOLD,
            len(preprocessor.columns_to_keep_),
            X_aug_train.shape[1] - len(preprocessor.columns_to_keep_),
            X_aug_train.shape[1],
        ],
    }
)

,metric,value
0,sparsity_threshold,0.9875
1,base_features_kept,2482.0000
2,added_rowwise_features,8.0000
3,final_features,2490.0000


In [7]:
model_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 8,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
    "min_split_gain": 0.0,
}

The ranking model is trained only on `X_aug_train`. Gain importance is used because it reflects the total objective improvement from splits using each feature.

In [8]:
ranking_model = build_lgbm_regressor(model_params)
ranking_model.fit(X_aug_train, y_train_log)

importance_df = pd.DataFrame(
    {
        "feature": X_aug_train.columns,
        "gain_importance": ranking_model.booster_.feature_importance(importance_type="gain"),
        "split_importance": ranking_model.booster_.feature_importance(importance_type="split"),
    }
).sort_values(
    ["gain_importance", "split_importance", "feature"],
    ascending=[False, False, True],
    ignore_index=True,
)

importance_df.head(20)

,feature,gain_importance,split_importance
0,nz_mean,32623.731701,532
1,nz_std,5174.287425,389
2,non_zero_count,3910.155883,348
3,row_max,3423.716172,244
4,f190486d6,2728.363044,138
5,row_sum,2504.297899,265
6,row_std,2069.473933,254
7,non_zero_ratio,1731.652779,107
8,15ace8c9f,865.359444,102
9,9fd594eec,724.808066,104


## Feature Importance Charts

The ranking model is trained on the training split only. Gain importance is the primary ranking signal, while split importance shows how often features are used in tree splits.

In [ ]:
TOP_N_IMPORTANCE = 30
top_gain_importance = importance_df.head(TOP_N_IMPORTANCE).sort_values("gain_importance")

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top_gain_importance["feature"], top_gain_importance["gain_importance"])
ax.set_title(f"Top {TOP_N_IMPORTANCE} LGBM Features By Gain Importance")
ax.set_xlabel("Gain importance")
ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
top_split_importance = importance_df.sort_values(
    ["split_importance", "gain_importance", "feature"],
    ascending=[False, False, True],
    ignore_index=True,
).head(TOP_N_IMPORTANCE).sort_values("split_importance")

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top_split_importance["feature"], top_split_importance["split_importance"])
ax.set_title(f"Top {TOP_N_IMPORTANCE} LGBM Features By Split Importance")
ax.set_xlabel("Split importance")
ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [9]:
def evaluate_top_k(top_k: int) -> dict[str, float | int]:
    selected_features = importance_df.head(top_k)["feature"].tolist()
    X_top_train = X_aug_train.loc[:, selected_features]
    X_top_test = X_aug_test.loc[:, selected_features]

    model = build_lgbm_regressor(model_params)
    cv_scores = -cross_val_score(
        estimator=model,
        X=X_top_train,
        y=y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    )

    model.fit(X_top_train, y_train_log)
    y_pred_log = model.predict(X_top_test)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)

    return {
        "top_k": top_k,
        "n_features": X_top_train.shape[1],
        "cv_rmsle_mean": cv_scores.mean(),
        "cv_rmsle_std": cv_scores.std(),
        "test_rmsle": root_mean_squared_log_error(y_test_raw, y_pred),
        "test_rmse": root_mean_squared_error(y_test_raw, y_pred),
        "test_mae": mean_absolute_error(y_test_raw, y_pred),
        "test_r2": r2_score(y_test_raw, y_pred),
    }

In [ ]:
top_k_results_df = pd.DataFrame(evaluate_top_k(top_k) for top_k in TOP_K_VALUES)
top_k_results_df.style.format(
    {
        "cv_rmsle_mean": "{:,.4f}",
        "cv_rmsle_std": "{:,.4f}",
        "test_rmsle": "{:,.4f}",
        "test_rmse": "{:,.0f}",
        "test_mae": "{:,.0f}",
        "test_r2": "{:,.4f}",
    }
)

In [ ]:
selected_feature_sets = {
    top_k: importance_df.head(top_k)["feature"].tolist()
    for top_k in TOP_K_VALUES
}

{top_k: features[:10] for top_k, features in selected_feature_sets.items()}

{100: ['nz_mean',
  'nz_std',
  'non_zero_count',
  'row_max',
  'f190486d6',
  'row_sum',
  'row_std',
  'non_zero_ratio',
  '15ace8c9f',
  '9fd594eec'],
 250: ['nz_mean',
  'nz_std',
  'non_zero_count',
  'row_max',
  'f190486d6',
  'row_sum',
  'row_std',
  'non_zero_ratio',
  '15ace8c9f',
  '9fd594eec'],
 500: ['nz_mean',
  'nz_std',
  'non_zero_count',
  'row_max',
  'f190486d6',
  'row_sum',
  'row_std',
  'non_zero_ratio',
  '15ace8c9f',
  '9fd594eec']}

In [ ]:
best_top_k_row = top_k_results_df.sort_values("cv_rmsle_mean", ignore_index=True).iloc[0]
best_top_k = int(best_top_k_row["top_k"])
best_features = selected_feature_sets[best_top_k]
X_best_train = X_aug_train.loc[:, best_features]
X_best_test = X_aug_test.loc[:, best_features]

summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "feature_setup": "sparse_threshold_0.9875_plus_rowwise_top_gain_importance",
    "sparsity_threshold": SPARSITY_THRESHOLD,
    "source_features": X_aug_train.shape[1],
    "top_k_values": TOP_K_VALUES,
    "selected_by": "lowest cv_rmsle_mean",
    "best_top_k": best_top_k,
    "model_params": model_params,
    "results": top_k_results_df.to_dict(orient="records"),
}

(X_best_train.shape, X_best_test.shape, summary)

((2987, 250),
 (1472, 250),
 {'target_transform': 'log1p',
  'primary_metric': 'rmsle',
  'feature_setup': 'sparse_threshold_0.9875_plus_rowwise_top_gain_importance',
  'sparsity_threshold': 0.9875,
  'source_features': 2490,
  'top_k_values': [100, 250, 500],
  'selected_by': 'lowest cv_rmsle_mean',
  'best_top_k': 250,
  'model_params': {'n_estimators': 500,
   'learning_rate': 0.03,
   'num_leaves': 31,
   'max_depth': 8,
   'min_child_samples': 20,
   'subsample': 0.8,
   'subsample_freq': 1,
   'colsample_bytree': 0.8,
   'reg_alpha': 0.05,
   'reg_lambda': 0.05,
   'min_split_gain': 0.0},
  'results': [{'top_k': 100,
    'n_features': 100,
    'cv_rmsle_mean': 1.3416427424336308,
    'cv_rmsle_std': 0.03836602859331192,
    'test_rmsle': 1.4103384024883983,
    'test_rmse': 6948390.851140491,
    'test_mae': 3961880.3422118993,
    'test_r2': 0.24354278586579425},
   {'top_k': 250,
    'n_features': 250,
    'cv_rmsle_mean': 1.331283064098332,
    'cv_rmsle_std': 0.02632986683750

## How To Read The Results

- Compare `top_k` candidates by `cv_rmsle_mean`; use test metrics only as a final sanity check.
- If the scores are close, prefer the smaller feature set because it is simpler and faster.
- If every top-k set is worse than notebook `06`, keep the full sparse-filtered row-wise dataset for tuning.
- For a stricter estimate, move feature selection inside each CV fold or rerun this as nested CV; this notebook uses a train-only ranking model as a pragmatic experiment.

# 07 Feature-Importance Top-K LGBM Report

## Goal

The goal of this notebook was to test whether the sparse-filtered row-wise feature set from notebook `06` can be reduced to the most important LightGBM features without losing predictive quality.

## What Was Done

- Loaded `data/processed_data.csv`.
- Split the data into train and test parts using the same seed and test size as notebook `06`.
- Applied sparse filtering with threshold `0.9875` and added row-wise aggregate features.
- Built a source feature space with `2,490` features.
- Trained a LightGBM ranking model on the training split only.
- Ranked features by LightGBM gain importance, with split importance kept as a secondary diagnostic.
- Evaluated reduced datasets using the top `100`, `250`, and `500` gain-ranked features.

## Most Important Features

The top gain-importance features were dominated by row-wise aggregate features:

| rank | feature | gain importance | split importance |
| ---: | --- | ---: | ---: |
| 1 | `nz_mean` | 32,623.7317 | 532 |
| 2 | `nz_std` | 5,174.2874 | 389 |
| 3 | `non_zero_count` | 3,910.1559 | 348 |
| 4 | `row_max` | 3,423.7162 | 244 |
| 5 | `f190486d6` | 2,728.3630 | 138 |
| 6 | `row_sum` | 2,504.2979 | 265 |
| 7 | `row_std` | 2,069.4739 | 254 |
| 8 | `non_zero_ratio` | 1,731.6528 | 107 |
| 9 | `15ace8c9f` | 865.3594 | 102 |
| 10 | `9fd594eec` | 724.8081 | 104 |

## Top-K Results

| top-k | CV RMSLE mean | CV RMSLE std | Test RMSLE | Test RMSE | Test MAE | Test R2 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 100 | 1.3416 | 0.0384 | 1.4103 | 6,948,390.85 | 3,961,880.34 | 0.2435 |
| 250 | 1.3313 | 0.0263 | 1.4079 | 6,920,826.09 | 3,937,197.32 | 0.2495 |
| 500 | 1.3382 | 0.0263 | 1.3960 | 6,941,127.50 | 3,937,143.02 | 0.2451 |

The best candidate by CV RMSLE was `top_k=250`, so the notebook materialized `X_best_train` with shape `(2987, 250)` and `X_best_test` with shape `(1472, 250)`.

## Comparison To Notebook 06

Notebook `06` used all `2,490` sparse-filtered row-wise features and achieved CV RMSLE `1.3757` with test RMSLE `1.3851`.

The top-k feature sets improved CV RMSLE, especially `top_k=250` at `1.3313`, but all top-k variants had worse held-out test RMSLE than the full feature set from notebook `06`. The best held-out result among the top-k candidates was `top_k=500` with test RMSLE `1.3960`, still worse than `1.3851` from the full feature set.

## Conclusion

Feature-importance selection is useful diagnostically and confirms that the row-wise aggregate features are highly important. However, based on the held-out test metric, the reduced top-k datasets should not replace the full `2,490`-feature sparse-filtered row-wise setup for the next tuning step. The CV improvement looks optimistic, so a stricter version would move feature selection inside each CV fold or use nested CV before accepting a reduced feature set.